# 01a — Flood mapping with optical images: georeader, spectral signatures and water indices
**Event:** EMSR271_02FARKADONA — Pineios river, Thessaly (Greece), Sentinel-2 image of 28 Feb 2018.
This event is in the **test set** of the model we train later, so we can compare every method on it.

0. **georeader in 5 minutes**: rasters as `GeoTensor`s = numpy arrays that know where they are on Earth.
1. Look at the image: true colour vs false colour.
2. Spectral signatures of water, land, clouds and cloud shadows.
3. Water indices (NDWI, MNDWI, AWEI) and a threshold → a flood map.
4. Limits of optical images.

**Google Colab:** run the next cell first (~1 min). It downloads the course code, data and models and installs
ml4floods; if Colab disconnects, just run it again. On your own computer it does nothing.

In [ ]:
import sys
if "google.colab" in sys.modules:
    import subprocess
    from pathlib import Path
    COURSE = Path("/content/geomundus-ml4floods")
    if not COURSE.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/kipoju/geomundus-ml4floods", str(COURSE)], check=True)
    sys.path.insert(0, str(COURSE / "notebooks"))
    import colab_setup
    colab_setup.setup()

In [ ]:
import sys
sys.path.append(".")

import matplotlib.pyplot as plt
import numpy as np
from georeader import plot, read
from georeader.geotensor import GeoTensor
from georeader.rasterio_reader import RasterioReader
from georeader.save import save_cog
from matplotlib.patches import Rectangle
from ml4floods.visualization.plot_utils import plot_gt_v2
from course_utils import (BANDS_S2, BOX_BOUNDS, BOX_CRS, EVENT, INDEX_CMAP, ROOT, S2_WAVELENGTHS, awei_sh,
                          binary_metrics, composite, confusion_legend, confusion_overlay, find_event, hist_by_class,
                          interact, ndwi, read_event, save_fig)

data_dir, split = find_event(EVENT)

## 0. georeader in 5 minutes
[georeader](https://github.com/spaceml-org/georeader) (also from SpaceML, used inside ml4floods) reads and
writes rasters as **`GeoTensor`s**: a numpy array (`band, y, x`) plus the three things that place it on Earth:
- `transform`: pixel (row, col) → map coordinates (x, y): origin and pixel size,
- `crs`: the coordinate reference system of those coordinates (here UTM zone 34N),
- `fill_value_default`: the value of the pixels without data.

### Open a file without reading it
`RasterioReader` is **lazy**: it only reads the metadata. Nothing is loaded until we call `.load()` — handy
for big scenes or files on the cloud (it also accepts `https://` and `gs://` paths).

In [ ]:
reader = RasterioReader(str(data_dir / split / "S2" / f"{EVENT}.tif"))
print("dims:", reader.dims, "| shape:", reader.shape, "| pixel size:", reader.res, "m")
print("crs:", reader.crs, "| bounds (xmin, ymin, xmax, ymax):", reader.bounds)
print(reader.transform)

### Read some bands → a GeoTensor
`isel` selects bands (0-based) before reading; `.load()` reads the pixels.

In [ ]:
rgb = reader.isel({"band": [3, 2, 1]}).load()          # B4, B3, B2
print(type(rgb).__name__, "| is a numpy array:", isinstance(rgb, np.ndarray), "|", rgb.dtype, rgb.shape)
rgb

### The golden rule: write numpy, get GeoTensors back
A GeoTensor **is** a numpy array (a subclass of `np.ndarray`), so numpy arithmetic, comparisons and ufuncs
work as usual — and the result **keeps the georeference**. Write your functions for plain numpy arrays and
they work on georeferenced images for free:

In [ ]:
def to_display(x: np.ndarray, max_val: float = 3000.) -> np.ndarray:
    """Reflectance × 10⁴ → [0, 1] for plotting."""
    return np.clip(x / max_val, 0, 1)


rgb_display = to_display(rgb)
print(type(rgb_display).__name__, "| same grid:", rgb_display.transform == rgb.transform, rgb_display.crs)

fig, ax = plt.subplots(figsize=(8, 7))
plot.show(rgb_display, ax=ax, add_scalebar=True)   # georeader plot: lon/lat ticks + scale bar from the GeoTensor
ax.set_title("FARKADONA, Sentinel-2 true colour")
plt.show()

### Crop by coordinates, not by pixels
Cut a 10 × 5.6 km box around the river given in UTM metres (it could be lon/lat: `crs_bounds="EPSG:4326"`).
Works on the lazy reader (reads only that window) and on a GeoTensor already in memory.

In [ ]:
box = read.read_from_bounds(reader, BOX_BOUNDS, crs_bounds=BOX_CRS).load()
print("box shape:", box.shape, "| bounds:", box.bounds)
print("footprint in lon/lat:", box.footprint("EPSG:4326"))

Two things that do **not** keep the georeference (they return plain numpy):
- boolean indexing `g[mask]` (not supported: use `g.values[mask]`, e.g. to collect pixels for a histogram),
- functions that are not numpy ufuncs (`np.where`, `scipy`, `torch`…). Put the georeference back with
  `g.array_as_geotensor(result)` — we will do this with the deep learning model in notebook 02a.

### Save a result as a Cloud Optimised GeoTIFF
Any GeoTensor can be written with its georeference and opened in QGIS:

In [ ]:
out_dir = ROOT / "outputs"
out_dir.mkdir(exist_ok=True)
save_cog((rgb_display * 255).astype(np.uint8), str(out_dir / "farkadona_rgb.tif"), descriptions=["B4", "B3", "B2"])

From here on, `read_event` (in `course_utils.py`) reads every product of an event like this: Sentinel-2,
the reference flood map (`gt`), and later Sentinel-1 and HAND, all as GeoTensors **on the same grid**.

In [ ]:
d = read_event(data_dir, split, EVENT)
# gt (WorldFloods v2): band 0 = {0: invalid, 1: clear, 2: cloud}, band 1 = {0: invalid, 1: land, 2: water}
s2, gt = d["s2"], d["gt"]
print(EVENT, "| split:", split, "| S2 date:", d["meta"]["s2_date"], "| shape:", s2.shape,
      "| same grid as gt:", s2.same_extent(gt))

## 1. True colour vs false colour
**Question:** where is the water easier to spot? Why?

In [ ]:
rgb = composite(s2, ("B4", "B3", "B2"), 3000)
swir = composite(s2, ("B11", "B8", "B4"), 4500)

fig, axs = plt.subplots(1, 3, figsize=(20, 6))
axs[0].imshow(rgb);  axs[0].set_title("True colour (R, G, B) = (B4, B3, B2)")
axs[1].imshow(swir); axs[1].set_title("False colour (SWIR, NIR, Red) = (B11, B8, B4)")
plot_gt_v2(gt.values, ax=axs[2]); axs[2].set_title("Reference (Copernicus EMS + clouds)")
for ax in axs:
    ax.set_axis_off()
plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5.9))
axs[0].imshow(rgb);  axs[0].set_title("True colour  (R, G, B) = (B4, B3, B2)", fontsize=14)
axs[1].imshow(swir); axs[1].set_title("False colour  (SWIR, NIR, Red) = (B11, B8, B4)", fontsize=14)
for ax in axs:
    ax.set_axis_off()
plt.tight_layout()
save_fig(fig, "opt_composites")
plt.close(fig)

## 2. Spectral signatures
We use the reference mask to collect the pixels of each class and plot their median
top-of-atmosphere reflectance in every Sentinel-2 band (shaded: 25–75 % percentiles).
Cloud shadows are not in the WorldFloods reference: we take them from the CloudSEN12 cloud
detection model (Aybar et al., 2022).

**Predict first:** in which bands will water be darker than land? And cloud shadows?

In [ ]:
cloud = gt[0] == 2
classes = {"Water": ((gt[1] == 2) & ~cloud, "tab:blue"),
           "Land": ((gt[1] == 1) & ~cloud, "tab:orange"),
           "Cloud": (cloud, "tab:gray")}
if "cloudsen12" in d:  # cloud shadows over land (CloudSEN12 class 3)
    shadow = (d["cloudsen12"].values == 3) & (gt[1] == 1) & ~cloud
    classes = {**classes, "Cloud shadow": (shadow, "tab:green")}
refl = s2.values / 10_000

fig, ax = plt.subplots(figsize=(14, 5.9))
for name, (mask, color) in classes.items():
    q25, q50, q75 = np.percentile(refl[:, mask], [25, 50, 75], axis=1)
    ax.plot(q50, color=color, lw=2.5, marker="o", label=f"{name} ({mask.sum():,} px)")
    ax.fill_between(range(len(q50)), q25, q75, color=color, alpha=0.2)
for i in range(len(BANDS_S2)):
    ax.axvline(i, color="gray", ls="--", lw=0.6, alpha=0.7)
ax.set_xticks(range(len(BANDS_S2)), [f"{b}\n{w}" for b, w in zip(BANDS_S2, S2_WAVELENGTHS)], fontsize=11)
ax.set_xlabel("Sentinel-2 band / wavelength λ [nm]", fontsize=12)
ax.set_ylabel(r"Reflectance $\rho_{TOA}$", fontsize=12)
ax.set_ylim(bottom=0)
ax.legend(fontsize=12)
plt.tight_layout()
save_fig(fig, "opt_spectral_profiles")

Water reflects little in the visible and almost nothing in the NIR (B8) and SWIR (B11, B12):
that contrast with land is what the water indices exploit. **Cloud shadows** are dark in all bands,
also in NIR and SWIR → they can be confused with water. Clouds are bright everywhere →
nothing can be seen below them (look at the NE of the image).

## 3. Water indices
- **NDWI** (McFeeters, 1996) = (Green − NIR) / (Green + NIR)
- **MNDWI** (Xu, 2006) = (Green − SWIR1) / (Green + SWIR1)
- **AWEI<sub>sh</sub>** (Feyisa et al., 2014) = Blue + 2.5 Green − 1.5 (NIR + SWIR1) − 0.25 SWIR2

All three are > 0 where water is more reflective in the visible than in the infrared.
Each one is a few lines of numpy on the bands → applied to the GeoTensor, the index is a GeoTensor too:

In [ ]:
def mndwi(s2: np.ndarray) -> np.ndarray:
    """MNDWI = (Green - SWIR1) / (Green + SWIR1) of a 13-band Sentinel-2 image (band, y, x)."""
    green, swir1 = s2[BANDS_S2.index("B3")], s2[BANDS_S2.index("B11")]
    return (green - swir1) / (green + swir1 + 1e-6)


water_mndwi = mndwi(s2) > 0
print(type(water_mndwi).__name__, water_mndwi.dtype, "| same grid as the image:", water_mndwi.same_extent(s2))

In [ ]:
indices = {"NDWI": ndwi(s2), "MNDWI": mndwi(s2), "AWEI$_{sh}$": awei_sh(s2)}   # GeoTensors (H, W)
VLIM = {"NDWI": 0.6, "MNDWI": 0.6, "AWEI$_{sh}$": 0.3}
ref = gt[1] == 2                  # reference water (also under clouds)
valid = (gt[1] != 0) & ~cloud     # evaluate on valid, cloud-free pixels

fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (name, idx) in zip(axs, indices.items()):
    im = ax.imshow(idx, cmap=INDEX_CMAP, vmin=-VLIM[name], vmax=VLIM[name])
    ax.set_title(name, fontsize=14)
    ax.set_axis_off()
    fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
plt.tight_layout()
save_fig(fig, "opt_indices")

## 4. From index to flood map: choose a threshold
How well does each index separate the two classes? Histograms of each index for the reference
water and land pixels (clouds excluded), and the flood map obtained with a threshold of 0:
**navy** = water hit (TP), **red** = false alarm (FP), **green** = missed water (FN), grey = land hit.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(14, 6.4), gridspec_kw={"height_ratios": [1, 1.5]})
for j, (name, idx) in enumerate(indices.items()):
    lim = 2 * VLIM[name]
    hist_by_class(idx, ref & valid, ~ref & valid, t=0, ax=axs[0, j], bins=np.linspace(-lim, lim, 81), xlabel=name)
    axs[0, j].set_yscale("log")
    axs[0, j].legend(fontsize=7)
    pred = idx > 0
    m = binary_metrics(pred, ref, valid)
    axs[1, j].imshow(confusion_overlay(pred, ref, valid), interpolation="nearest")
    axs[1, j].set_title(f"{name} > 0 — P {m['precision']:.2f} · R {m['recall']:.2f} · IoU {m['IoU']:.2f}", fontsize=11)
    axs[1, j].set_axis_off()
confusion_legend(axs[1, 0], fontsize=7)
plt.tight_layout()
save_fig(fig, "opt_index_threshold")

**Predict first:** which threshold on MNDWI would you choose? Move the slider and look at the errors.

In [ ]:
def show_mndwi_threshold(t=0.0):
    pred = indices["MNDWI"] > t
    m = binary_metrics(pred, ref, valid)
    fig, axs = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.2, 1]})
    axs[0].imshow(confusion_overlay(pred, ref, valid), interpolation="nearest")
    axs[0].set_title(f"MNDWI > {t:.2f} — P={m['precision']:.2f}  R={m['recall']:.2f}  IoU={m['IoU']:.2f}")
    axs[0].set_axis_off()
    confusion_legend(axs[0])
    hist_by_class(indices["MNDWI"], ref & valid, ~ref & valid, t=t, ax=axs[1],
                  bins=np.linspace(-1, 1, 101), xlabel="MNDWI")
    axs[1].set_yscale("log")
    plt.tight_layout()
    plt.show()


import ipywidgets as widgets

interact(show_mndwi_threshold, t=widgets.FloatSlider(value=0.0, min=-0.5, max=0.5, step=0.02))

## 5. Limits of optical images
Clouds hide the flood; their shadows are dark like water; sediment-laden flood water is brown and
bright in the visible.

In [ ]:
W = 300
# panels in the same left-to-right order as the boxes on the map
windows = {"Turbid (sediment-laden) water": (900, 650), "Clouds and cloud shadows": (500, 1800)}
fig, axs = plt.subplots(1, 3, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1.22, 1, 1]})
axs[0].imshow(rgb)
axs[0].set_title("Sentinel-2 true colour", fontsize=13)
for ax, (name, (r, c)), col in zip(axs[1:], windows.items(), ["cyan", "yellow"]):
    axs[0].add_patch(Rectangle((c, r), W, W, lw=2, edgecolor=col, facecolor="none"))
    ax.imshow(composite(s2.values[:, r:r + W, c:c + W], ("B4", "B3", "B2"), 3000))
    ax.set_title(name, fontsize=13)
    for side in ax.spines.values():
        side.set_edgecolor(col)
        side.set_linewidth(4)
    ax.set_xticks([]); ax.set_yticks([])
axs[0].set_axis_off()
plt.tight_layout()
save_fig(fig, "opt_limits")

**Take-aways**
- One index + one threshold already gives a reasonable flood map on a clear-sky image.
- The best threshold changes from image to image (turbid water, shadows, snow, burnt areas…).
- Under clouds (NE of the image) optical sensors see nothing → next notebook: **SAR**.